In [ ]:
FIGURE = 'Fig.S10'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
ssmi_file = asset_path('asset_174bc324d489')
spei_file = asset_path('asset_beb53c9b3525')
out_dir = asset_path('asset_0517d1522006')
os.makedirs(out_dir, exist_ok=True)
out_png = os.path.join(out_dir, 'Fig_single_pixel_SPEI_vs_SSMIrz_scale_redundancy_36p2N_115p2E.png')
target_lat = 50.25
target_lon = 10.25

def get_var(ds, preferred_names):
    for name in preferred_names:
        if name in ds.data_vars:
            return ds[name]
    for name in ds.data_vars:
        lname = name.lower()
        if any((key in lname for key in ['ssmi', 'spei'])):
            return ds[name]
    raise ValueError(f'Cannot find variable. Actual variables: {list(ds.data_vars)}')

def standardize_coord_names(da):
    rename_dict = {}
    for d in da.dims:
        dl = d.lower()
        if dl in ['latitude', 'y']:
            rename_dict[d] = 'lat'
        elif dl in ['longitude', 'x']:
            rename_dict[d] = 'lon'
        elif dl in ['month', 'date']:
            rename_dict[d] = 'time'
    if rename_dict:
        da = da.rename(rename_dict)
    return da

def pearsonr_1d(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if x.size != y.size:
        raise ValueError(f'Input lengths are different: x={x.size}, y={y.size}')
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan
    return np.float32(np.corrcoef(x, y)[0, 1])

def scale_corr_matrix_at_pixel(da_pixel):
    if 'scale' not in da_pixel.dims:
        raise ValueError(f"Input DataArray has no 'scale' dimension: {da_pixel.dims}")
    if 'time' not in da_pixel.dims:
        raise ValueError(f"Input DataArray has no 'time' dimension: {da_pixel.dims}")
    da_pixel = da_pixel.sortby('scale')
    scales = da_pixel['scale'].values
    nscale = len(scales)
    mat = np.full((nscale, nscale), np.nan, dtype=np.float32)
    for i in range(nscale):
        x = da_pixel.isel(scale=i).values
        for j in range(nscale):
            y = da_pixel.isel(scale=j).values
            mat[i, j] = pearsonr_1d(x, y)
    return (scales, mat)

def find_scale_tick_positions(scales, requested_ticks):
    scales = np.asarray(scales)
    tick_positions = []
    tick_labels = []
    for scale_value in requested_ticks:
        matched = np.where(np.isclose(scales.astype(np.float64), float(scale_value)))[0]
        if matched.size > 0:
            tick_positions.append(int(matched[0]))
            tick_labels.append(str(scale_value))
    return (tick_positions, tick_labels)
print('Loading SSMIrz...')
ds_ssmi = xr.open_dataset(ssmi_file, engine='h5netcdf')
ssmi = get_var(ds_ssmi, ['SSMIrz', 'ssmirz', 'SSMI', 'ssmi'])
ssmi = standardize_coord_names(ssmi)
print('Loading SPEI...')
ds_spei = xr.open_dataset(spei_file, engine='h5netcdf')
spei = get_var(ds_spei, ['SPEI', 'spei'])
spei = standardize_coord_names(spei)
for da_name, da in [('SSMIrz', ssmi), ('SPEI', spei)]:
    for required_dim in ['scale', 'time', 'lat', 'lon']:
        if required_dim not in da.dims:
            raise ValueError(f"{da_name} does not contain dimension '{required_dim}'. Actual dims: {da.dims}")
ssmi = ssmi.sortby('scale').sortby('time')
spei = spei.sortby('scale').sortby('time')
print('\nData information:')
print(f'SSMIrz dimensions: {ssmi.dims}')
print(f'SSMIrz shape: {ssmi.shape}')
print(f'SPEI dimensions: {spei.dims}')
print(f'SPEI shape: {spei.shape}')
ssmi_pix = ssmi.sel(lat=target_lat, lon=target_lon, method='nearest').sel(scale=slice(1, 24))
spei_pix = spei.sel(lat=target_lat, lon=target_lon, method='nearest').sel(scale=slice(1, 24))
actual_lat_ssmi = float(ssmi_pix['lat'].values)
actual_lon_ssmi = float(ssmi_pix['lon'].values)
actual_lat_spei = float(spei_pix['lat'].values)
actual_lon_spei = float(spei_pix['lon'].values)
print('\nPixel selection:')
print(f'Requested pixel: lat={target_lat}, lon={target_lon}')
print(f'SSMIrz nearest pixel: lat={actual_lat_ssmi}, lon={actual_lon_ssmi}')
print(f'SPEI nearest pixel: lat={actual_lat_spei}, lon={actual_lon_spei}')
ssmi_1 = ssmi_pix.sel(scale=1)
ssmi_24 = ssmi_pix.sel(scale=24)
spei_1 = spei_pix.sel(scale=1)
spei_24 = spei_pix.sel(scale=24)
ssmi_1 = ssmi_1.squeeze(drop=True)
ssmi_24 = ssmi_24.squeeze(drop=True)
spei_1 = spei_1.squeeze(drop=True)
spei_24 = spei_24.squeeze(drop=True)
r_ssmi_1_24 = pearsonr_1d(ssmi_1.values, ssmi_24.values)
r_spei_1_24 = pearsonr_1d(spei_1.values, spei_24.values)
print('\nCorrelations between 1-month and 24-month scales:')
print(f'Single-pixel corr SSMIrz-1 vs SSMIrz-24: {r_ssmi_1_24:.4f}')
print(f'Single-pixel corr SPEI-1 vs SPEI-24: {r_spei_1_24:.4f}')
print('\nComputing single-pixel SPEI scale correlation matrix...')
spei_scales, spei_mat = scale_corr_matrix_at_pixel(spei_pix)
print('Computing single-pixel SSMIrz scale correlation matrix...')
ssmi_scales, ssmi_mat = scale_corr_matrix_at_pixel(ssmi_pix)
print('SPEI correlation matrix min/max:', np.nanmin(spei_mat), np.nanmax(spei_mat))
print('SSMIrz correlation matrix min/max:', np.nanmin(ssmi_mat), np.nanmax(ssmi_mat))
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 10
math_delimiter = chr(36)
SSMI_LABEL = f'{math_delimiter}\\mathrm{{SSMI}}_{{\\mathrm{{rz}}}}{math_delimiter}'
fig = plt.figure(figsize=(12, 8.8), dpi=600)
gs = fig.add_gridspec(2, 2, width_ratios=[1.35, 1.0], height_ratios=[1.0, 1.0], wspace=0.28, hspace=0.34)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(spei_1['time'].values, spei_1.values, color='#E69F00', linewidth=1.1, alpha=0.85, label='SPEI-1')
ax1.plot(spei_24['time'].values, spei_24.values, color='#D55E00', linewidth=2.0, alpha=0.95, label='SPEI-24')
ax1.axhline(0, color='black', linewidth=0.8, alpha=0.75)
ax1.set_ylabel('SPEI')
ax1.set_title(f'a  SPEI-1 vs SPEI-24, r = {r_spei_1_24:.2f}', loc='left', fontsize=11, pad=8)
ax1.legend(frameon=False, ncol=2, loc='upper right')
ax1.grid(True, axis='y', linestyle='-', linewidth=0.35, color='#E6E6E6')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(spei_mat, origin='lower', cmap='RdBu_r', vmin=-1, vmax=1, interpolation='nearest', aspect='equal')
tick_scales = [1, 6, 12, 18, 24]
tick_pos_spei, tick_lab_spei = find_scale_tick_positions(spei_scales, tick_scales)
ax2.set_xticks(tick_pos_spei)
ax2.set_xticklabels(tick_lab_spei)
ax2.set_yticks(tick_pos_spei)
ax2.set_yticklabels(tick_lab_spei)
ax2.set_xlabel('SPEI timescale / months')
ax2.set_ylabel('SPEI timescale / months')
ax2.set_title('b  SPEI scale correlation', loc='left', fontsize=11, pad=8)
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.035)
cbar2.set_label('Pearson r', fontsize=9)
cbar2.ax.tick_params(labelsize=8.5)
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(ssmi_1['time'].values, ssmi_1.values, color='#56B4E9', linewidth=1.1, alpha=0.85, label=f'{SSMI_LABEL}-1')
ax3.plot(ssmi_24['time'].values, ssmi_24.values, color='#0072B2', linewidth=2.0, alpha=0.95, label=f'{SSMI_LABEL}-24')
ax3.axhline(0, color='black', linewidth=0.8, alpha=0.75)
ax3.set_ylabel(SSMI_LABEL)
ax3.set_title(f'c  {SSMI_LABEL}-1 vs {SSMI_LABEL}-24, r = {r_ssmi_1_24:.2f}', loc='left', fontsize=11, pad=8)
ax3.legend(frameon=False, ncol=2, loc='upper right')
ax3.grid(True, axis='y', linestyle='-', linewidth=0.35, color='#E6E6E6')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax4 = fig.add_subplot(gs[1, 1])
im4 = ax4.imshow(ssmi_mat, origin='lower', cmap='RdBu_r', vmin=-1, vmax=1, interpolation='nearest', aspect='equal')
tick_pos_ssmi, tick_lab_ssmi = find_scale_tick_positions(ssmi_scales, tick_scales)
ax4.set_xticks(tick_pos_ssmi)
ax4.set_xticklabels(tick_lab_ssmi)
ax4.set_yticks(tick_pos_ssmi)
ax4.set_yticklabels(tick_lab_ssmi)
ax4.set_xlabel(f'{SSMI_LABEL} timescale / months')
ax4.set_ylabel(f'{SSMI_LABEL} timescale / months')
ax4.set_title(f'd  {SSMI_LABEL} scale correlation', loc='left', fontsize=11, pad=8)
cbar4 = plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.035)
cbar4.set_label('Pearson r', fontsize=9)
cbar4.ax.tick_params(labelsize=8.5)
fig.suptitle(f'Single-pixel comparison of scale redundancy in SPEI and {SSMI_LABEL}\nNearest grid cell: {SSMI_LABEL} ({actual_lat_ssmi:.2f}°N, {actual_lon_ssmi:.2f}°E), SPEI ({actual_lat_spei:.2f}°N, {actual_lon_spei:.2f}°E)', fontsize=12.5, x=0.08, y=0.985, ha='left')
plt.savefig(out_png, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print(f'\nSaved single-pixel comparison figure:\n{out_png}')
ds_ssmi.close()
ds_spei.close()
